In [1]:
using MarineHydro
using PyCall
using DimensionalData
using Test
using DifferentiationInterface 
import ForwardDiff 
import Zygote
# import Enzyme
# import Mooncake
using FiniteDifferences
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
betas = [0.0, pi/6] # incident wave angle [rad]
beta = betas[1]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Capytaine Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)



PyObject Mesh(vertices=[[... 83 vertices ...]], faces=[[... 80 faces ...]], name="cylinder_0")

In [11]:
@testset "Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder" begin
    
    # Description of problem
    omegas = [1.0, 1.5] # frequencies [rad/s]
    beta = 0 # incident wave angle [rad]
    t_DOFs = ["Heave"] # translational DOFs
    r_DOFs = ["Pitch"] # rotational DOFs
    DOFs = [t_DOFs; r_DOFs] # all DOFs

    # Create Mesh object
    radius = 1.5  
    center = (0.0,0.0,0.0) 
    len = 2.5
    faces_max_radius = 0.7
    cpt = pyimport("capytaine")
    cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
                radius=radius,
                center=center, 
                length=len, 
                faces_max_radius = faces_max_radius
                ).keep_immersed_part(inplace=true)

    # Get MarineHydro values
    mesh = Mesh(cptmesh)
    rigid_dof_list = DOFs
    rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
    floatingbody = FloatingBody(mesh, rigid_dof_list, rotation_center, "Horizontal_Cylinder")

    # Radiation solve functions
    function A_and_B_vec(w)
        parameters = (wave_frequencies=w,
            radiating_dofs=collect(keys(floatingbody.dofs)),
            influenced_dofs=collect(keys(floatingbody.dofs)))
        data = compute_hydrodynamic_coefficients(parameters, floatingbody)
        return vcat(vec(data.added_mass), vec(data.radiation_damping)) 
    end

    # Incident + diffraction solve function
    function F_ex_vec(w)
        parameters = (wave_frequencies=[w],
            wave_directions=[beta],
            influenced_dofs=collect(keys(floatingbody.dofs)))
        data = compute_hydrodynamic_coefficients(parameters, floatingbody)
        return vcat(real.(vec(data.excitation_force)),imag.(vec(data.excitation_force))) 
    end

    backend = AutoForwardDiff()



    for omega in omegas
        @testset "Verify sensitivity of added mass and damping wrt Omega: $omega" begin
            A_and_B_FD = FiniteDifferences.central_fdm(5, 1)(A_and_B_vec, omega)
            A_and_B_AD = derivative(A_and_B_vec, backend, omega)
            @test A_and_B_AD !== nothing
            @test typeof(A_and_B_AD) == Vector{Float64}
            @test A_and_B_AD ≈ A_and_B_FD atol=1e-6 rtol=1e-6
        end
        @testset "Verify sensitivity of excitation force wrt Omega: $omega" begin
            F_ex_FD = FiniteDifferences.central_fdm(5, 1)(F_ex_vec, omega)
            F_ex_AD = derivative(F_ex_vec, backend, omega)
            @test F_ex_AD !== nothing
            @test typeof(F_ex_AD) == Vector{Float64}
            @test F_ex_AD ≈ F_ex_FD atol=1e-6 rtol=1e-6
        end
    end  
end








Test Summary:                                                           | Pass  Total     Time
Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder |   12     12  1m42.9s


Test.DefaultTestSet("Gradient accuracy check with Finite diff [w.r.t omega] for MDOF cylnder", Any[Test.DefaultTestSet("Verify sensitivity of added mass and damping wrt Omega: 1.0", Any[], 3, false, false, true, 1.775144386862e9, 1.775144431292e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl"), Test.DefaultTestSet("Verify sensitivity of excitation force wrt Omega: 1.0", Any[], 3, false, false, true, 1.775144431292e9, 1.775144465275e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl"), Test.DefaultTestSet("Verify sensitivity of added mass and damping wrt Omega: 1.5", Any[], 3, false, false, true, 1.775144465288e9, 1.775144481724e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X35sZmlsZQ==.jl"), Test.DefaultTestSet("Verify sensitivity of excitation force wrt Omega: 1.5", Any[], 3, false, false, true, 1.775

In [10]:
backend = AutoForwardDiff()
val, jac_tuple = value_and_derivative(A_and_B_vec, backend, 1.5)
display(jac_tuple)
jac_tuple = derivative(A_and_B_vec, backend, 1.5)
display(jac_tuple)

8-element Vector{Float64}:
 -2468.578125915292
 -2468.578125915289
 -2468.578125915292
 -2174.1445030194536
  2825.0875487937183
  2825.0875487937196
  2825.087548793718
  3140.9442643374105

8-element Vector{Float64}:
 -2468.578125915292
 -2468.578125915289
 -2468.578125915292
 -2174.1445030194536
  2825.0875487937183
  2825.0875487937196
  2825.087548793718
  3140.9442643374105

In [2]:
# Solve using Capytaine

# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = center
cptbody.rotation_center = rotation_center
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder 1"

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => DOFs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")

# Get Capytaine values
# A_cpt = cptresults.added_mass
# B_cpt = cptresults.radiation_damping
# F_FK_cpt = cptresults.Froude_Krylov_force 
# F_D_cpt = cptresults.diffraction_force
# F_ex_cpt = cptresults.excitation_force

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 740B
Dimensions:              (omega: 2, radiating_dof: 2, influenced_dof: 2,
                          wave_direction: 2)
Coordinates:
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    body_name            <U21 84B 'Horizontal Cylinder 1'
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
  * wave_direction       (wave_direction) float64 16B 0.0 0.5236
  * omega                (omega) float64 16B 1.03 1.7
  * radiating_dof        (radiating_dof) object 16B 'Heave' 'Pitch'
  * influenced_dof       (influenced_dof) object 16B 'Heave' 'Pitch'
    period               (omega) float64 16B 6.1 3.696
    wavenumber           (omega) float64 16B 0.1081 0.2946
    wavelength           (omega) float64 16B 58.1 21.33
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 64B 6...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 64B 1...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 128B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 128B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 128B ...
Attributes: (12/16)
    start_of_computation:                     2026-04-02T11:58:58.746452
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    gf_singularities:                         low_freq
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    creation_of_dataset:                      2026-04-02T11:58:58.758454
    capytaine_version:                        2.2.1

In [2]:
using Zygote
omegas = [1.0, 1.5] # frequencies [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.7
cpt = pyimport("capytaine")
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)

# Get MarineHydro values
mesh = Mesh(cptmesh)
rigid_dof_list = DOFs
rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
floatingbody = FloatingBody(mesh, rigid_dof_list, rotation_center, "Horizontal_Cylinder")

# Radiation solve functions
function A_and_B_vec(w)
    parameters = (wave_frequencies=w,
        radiating_dofs=collect(keys(floatingbody.dofs)),
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(vec(data.added_mass), vec(data.radiation_damping)) 
end
function Jacobian_of_rad_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        A_and_B_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

# Incident + diffraction solve function
function F_ex_vec(w)
    parameters = (wave_frequencies=[w],
        wave_directions=[beta],
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(real.(vec(data.excitation_force)),imag.(vec(data.excitation_force))) 
end
function Jacobian_of_dif_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        F_ex_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

Jacobian_of_dif_problem (generic function with 1 method)

In [5]:
mesh 

Mesh([-1.25 -1.5 2.755455298081545e-16; -1.25 -1.0606601717798214 -1.060660171779821; … ; 1.25 1.0606601717798214 -1.0606601717798214; 1.25 1.5 -9.184850993605148e-17], [13 4 9 14; 18 13 14 19; … ; 63 65 66 64; 57 59 58 56], [-1.125 0.5303300858899105 -1.2803300858899107; -0.8749999999999999 0.5303300858899105 -1.2803300858899107; … ; 1.25 0.995812289025486 -0.41247895569215276; 1.2500000000000002 -0.9958122890254862 -0.4124789556921525], [0.0 0.3826834323650895 -0.923879532511287; 0.0 0.3826834323650895 -0.923879532511287; … ; 1.0 0.0 -0.0; 1.0 0.0 -0.0], [0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743  …  0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261493, 0.19887378220871652, 0.19887378220871652, 0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261494], [0.5874775494988

In [3]:
floatingbody2 = FloatingBody(mesh, rigid_dof_list, rotation_center, "Horizontal_Cylinder_2")
floatingbody3 = floatingbody + floatingbody

FloatingBody(Mesh([-1.25 -1.5 2.755455298081545e-16; -1.25 -1.0606601717798214 -1.060660171779821; … ; 1.25 1.0606601717798214 -1.0606601717798214; 1.25 1.5 -9.184850993605148e-17], [13 4 9 14; 18 13 14 19; … ; 130 132 133 131; 124 126 125 123], [-1.125 0.5303300858899105 -1.2803300858899107; -0.8749999999999999 0.5303300858899105 -1.2803300858899107; … ; 1.25 0.995812289025486 -0.41247895569215276; 1.2500000000000002 -0.9958122890254862 -0.4124789556921525], [0.0 0.3826834323650895 -0.923879532511287; 0.0 0.3826834323650895 -0.923879532511287; … ; 1.0 0.0 -0.0; 1.0 0.0 -0.0], [0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743  …  0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261493, 0.19887378220871652, 0.19887378220871652, 0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.59662134662614

In [5]:
F_ex_vec(1.5)

4-element Vector{Float64}:
 45606.44550191805
 45338.29304007763
 -5702.900163657982
  2805.7681965590573

In [6]:
# Jacobian_of_dif_problem(1.5)

In [28]:
# Solve using MarineHydro

# Create Mesh struct
mhmesh = Mesh(cptmesh)

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=Symbol.(DOFs))

# Compute all hydrodynamic coefficients
mh_hydro_coeffs = compute_hydrodynamic_coefficients(parameters, floatingbody1)


(added_mass = [7153.377453259439 364822.2501162315; 364822.2501162314 1.860871333193322e7;;; 5998.050875879479 305900.5946698534; 305900.5946698531 1.560381953198017e7], radiation_damping = [1841.8271688251803 93933.1856100842; 93933.1856100842 4.79059617015096e6;;; 3647.557710560446 186025.4432385829; 186025.4432385828 9.487364288988056e6], diffraction_force = ComplexF64[-6739.4297299003565 - 1858.3255491107673im -11480.166268641755 - 5702.900163657982im; -343722.98890752037 - 92007.02585612508im -585756.6321625698 - 284509.77177990234im;;; -6796.718113295343 - 1855.8491308910513im -11741.061669824983 - 5664.105942849509im; -346643.08508832124 - 92246.86450907079im -599027.1632543269 - 283330.058786157im], Froude_Krylov_force = ComplexF64[65898.53109953046 + 4.263256414560601e-14im 57086.61177055981 + 1.1368683772161603e-13im; 3.3608250860760524e6 + 963.1129972945014im 2.911417200298552e6 + 2170.531793562212im;;; 65882.71133921918 - 4.973799150320701e-14im 57007.23003938994 + 5.684341

In [29]:
parameters.radiating_dofs

2-element Vector{Symbol}:
 :Heave
 :Pitch

In [30]:
# Create DimStack (not differentiable)
mhresults = compute_and_label_hydrodynamic_coefficients(parameters, floatingbody1)

┌ 2×2×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.0, 1.5] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :Froude_Krylov_force eltype: ComplexF64 dims: influenced_

In [11]:
# Compare added_mass
omega_num = 2

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

"Capytaine Added Mass"

PyCall.PyError: PyError ($(Expr(:escape, :(ccall(#= C:\Users\15183\.julia\packages\PyCall\1gn3u\src\pyfncall.jl:43 =# @pysym(:PyObject_Call), PyPtr, (PyPtr, PyPtr, PyPtr), o, pyargsptr, kw))))) <class 'KeyError'>
KeyError("not all values found in index 'omega'. Try setting the `method` keyword argument (example: method='nearest').")
  File "C:\Users\15183\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\core\dataarray.py", line 1681, in sel
    ds = self._to_temp_dataset().sel(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\15183\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\core\dataset.py", line 3242, in sel
    query_results = map_index_queries(
                    ^^^^^^^^^^^^^^^^^^
  File "C:\Users\15183\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\core\indexing.py", line 195, in map_index_queries
    results.append(index.sel(labels, **options))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\15183\App

In [12]:
# Multiple Body Case

# Create Capytaine Mesh object
radius = 1.5  
sep_dis = 50.0
center = (sep_dis,sep_dis,0.0) 
rotation_center = (sep_dis + 1.0, sep_dis + 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh2 = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody2 = cpt.FloatingBody(mesh=cptmesh2)
cptbody2.center_of_mass = center
cptbody2.rotation_center = rotation_center
foreach(dof -> cptbody2.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody2.add_rotation_dof(name=dof), r_DOFs)
cptbody2.active_dofs = DOFs
cptbody2.name = "Horizontal Cylinder 2"

# Create Mesh struct
mhmesh2 = Mesh(cptmesh2)

# Create FloatingBody struct
floatingbody2 = FloatingBody(mhmesh2,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 2")

floatingbodylist = [floatingbody1, floatingbody2]


2-element Vector{FloatingBody}:
 FloatingBody(Mesh([-1.25 -1.5 2.755455298081545e-16; -1.25 -1.0606601717798214 -1.060660171779821; … ; 1.25 1.0606601717798214 -1.0606601717798214; 1.25 1.5 -9.184850993605148e-17], [13 4 9 14; 18 13 14 19; … ; 63 65 66 64; 57 59 58 56], [-1.125 0.5303300858899105 -1.2803300858899107; -0.8749999999999999 0.5303300858899105 -1.2803300858899107; … ; 1.25 0.995812289025486 -0.41247895569215276; 1.2500000000000002 -0.9958122890254862 -0.4124789556921525], [0.0 0.3826834323650895 -0.923879532511287; 0.0 0.3826834323650895 -0.923879532511287; … ; 1.0 0.0 -0.0; 1.0 0.0 -0.0], [0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743, 0.28701257427381743  …  0.19887378220871646, 0.19887378220871652, 0.5966213466261495, 0.5966213466261493, 0.19887378220871652, 0.19887378220871652, 0.19887378220871646, 0.19887378220871652, 0.596621346

In [13]:
# Setup and solve BEM problems
cptarray = cptbody + cptbody2

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptarray.dofs
xr = pyimport("xarray")
rad_dofs = string.(collect(keys(dof_list)))
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => rad_dofs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptarray, method="direct")



Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 2kB
Dimensions:              (omega: 2, radiating_dof: 4, influenced_dof: 4,
                          wave_direction: 2)
Coordinates:
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    body_name            <U43 172B 'Horizontal Cylinder 1+Horizontal Cylinder 2'
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
  * wave_direction       (wave_direction) float64 16B 0.0 0.5236
  * omega                (omega) float64 16B 1.0 1.5
  * radiating_dof        (radiating_dof) object 32B 'Horizontal Cylinder 1__H...
  * influenced_dof       (influenced_dof) object 32B 'Horizontal Cylinder 1__...
    period               (omega) float64 16B 6.283 4.189
    wavenumber           (omega) float64 16B 0.1019 0.2294
    wavelength           (omega) float64 16B 61.64 27.39
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 256B ...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 256B ...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 256B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 256B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 256B ...
Attributes: (12/16)
    start_of_computation:                     2026-03-30T14:23:12.814142
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    gf_singularities:                         low_freq
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    creation_of_dataset:                      2026-03-30T14:23:12.833562
    capytaine_version:                        2.2.1

In [14]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=collect(keys(fb_array.dofs)))

# Create DimStack of results
mhresults = compute_and_label_hydrodynamic_coefficients(parameters, fb_array)

┌ 4×4×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Horizontal_Cylinder_1__Heave, …, :Horizontal_Cylinder_2__Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Horizontal_Cylinder_1__Heave, …, :Horizontal_Cylinder_2__Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.0, 1.5] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 4×4×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 4×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_fr

In [15]:
keys(cptarray.dofs)

KeySet for a Dict{Any, Any} with 4 entries. Keys:
  "Horizontal Cylinder 1__Heave"
  "Horizontal Cylinder 2__Heave"
  "Horizontal Cylinder 1__Pitch"
  "Horizontal Cylinder 2__Pitch"

In [16]:
cptresults.added_mass

PyObject <xarray.DataArray 'added_mass' (omega: 2, radiating_dof: 4, influenced_dof: 4)> Size: 256B
array([[[ 6.87974436e+03,  6.87984559e+03, -3.00581619e+01,
         -6.28035612e+01],
        [ 6.87983971e+03,  1.00066845e+04,  7.82370917e-01,
         -3.22344802e+01],
        [-3.00581619e+01,  2.68723748e+00,  6.87974436e+03,
          6.87964313e+03],
        [-6.08986946e+01, -2.84247471e+01,  6.87964901e+03,
          1.00062914e+04]],

       [[ 5.76387972e+03,  5.76200316e+03, -2.18388871e+02,
         -1.38516856e+02],
        [ 5.76211007e+03,  9.04168882e+03, -2.93723956e+02,
         -2.17051126e+02],
        [-2.18388871e+02, -2.98260886e+02,  5.76387972e+03,
          5.76575627e+03],
        [-1.43053785e+02, -2.26124984e+02,  5.76564936e+03,
          9.04898122e+03]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U43 172B 'Horizontal Cylinder 1+Horizontal Cylinder 2'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * omega           (omega) float64 16B 1.0 1.5
  * radiating_dof   (radiating_dof) object 32B 'Horizontal Cylinder 1__Heave'...
  * influenced_dof  (influenced_dof) object 32B 'Horizontal Cylinder 1__Heave...
    period          (omega) float64 16B 6.283 4.189
    wavenumber      (omega) float64 16B 0.1019 0.2294
    wavelength      (omega) float64 16B 61.64 27.39
Attributes:
    long_name:  Added mass

In [17]:
DOFz=String.(collect(keys(fb_array.dofs)))

4-element Vector{String}:
 "Horizontal_Cylinder_1__Heave"
 "Horizontal_Cylinder_1__Pitch"
 "Horizontal_Cylinder_2__Heave"
 "Horizontal_Cylinder_2__Pitch"

In [18]:
keys(fb_array.dofs)

(:Horizontal_Cylinder_1__Heave, :Horizontal_Cylinder_1__Pitch, :Horizontal_Cylinder_2__Heave, :Horizontal_Cylinder_2__Pitch)

In [19]:
mhresults.added_mass[influenced_dofs = At(Symbol(DOFz[1])),radiating_dofs = At(Symbol(DOFz[2])), wave_frequencies = At(omegas[1])]

7159.346194668676

In [20]:
# Compare added_mass
omega_num = 1

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

"Capytaine Added Mass"

4×4 Matrix{Float64}:
 6879.74     6879.85      -30.0582      -62.8036
 6879.84    10006.7         0.782371    -32.2345
  -30.0582      2.68724  6879.74       6879.64
  -60.8987    -28.4247   6879.65      10006.3

"MarineHydro Added Mass"

4×4 Matrix{Float64}:
 7159.27    7159.35      -30.2573     -62.1721
 7159.35    9937.99       -6.29128    -38.4012
  -30.2822    -7.15771  6882.45      6882.35
  -64.1976   -41.2727   6882.34     10008.5

In [21]:
rad_dofs[1]

"Horizontal Cylinder 1__Heave"

In [22]:
rad_dofs[1:3]

3-element Vector{String}:
 "Horizontal Cylinder 1__Heave"
 "Horizontal Cylinder 2__Heave"
 "Horizontal Cylinder 1__Pitch"

In [23]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

rad_dofs = collect(keys(fb_array.dofs))

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[rad_dofs[1]],
influenced_dofs=rad_dofs[1:3])

# Create DimStack of results
problems = problems_from_data(parameters, fb_array)
results = solve_all_problems(problems)
data = assemble_hydrodynamic_coefficients(parameters, fb_array, results)
# mhresults = compute_hydrodynamic_coefficients(parameters, fb_array)

(added_mass = [7159.272600749387; 7159.348934008887; -30.282202998346488;;;], radiation_damping = [1842.8796013136957; 1842.6151713714987; 527.1167487794835;;;], diffraction_force = ComplexF64[-7300.0097030348 - 2131.8675441657283im; -7326.884858992253 + 660.2855777420644im; -4010.913631094696 + 4867.826619099737im;;;], Froude_Krylov_force = ComplexF64[65898.53109953046 + 4.263256414560601e-14im; 65898.53109953046 + 963.1129972944948im; 23832.38451226816 - 58906.04628505189im;;;], excitation_force = ComplexF64[58598.521396495664 - 2131.8675441657283im; 58571.646240538204 + 1623.3985750365591im; 19821.470881173464 - 54038.21966595215im;;;])

In [24]:
# Single dof differentiability

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    [:Heave],
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[:Heave])


function get_added_mass(omega)
    parameters = (wave_frequencies=[omega], 
    wave_directions=[betas[1]],
    radiating_dofs=[:Heave])
    mhresults = compute_hydrodynamic_coefficients(parameters, floatingbody1)
    added_mass = mhresults.added_mass[1]
    return added_mass
end


get_added_mass(1.5)

5998.050875879479

In [25]:
using Zygote

In [26]:

# A_w_grad, = Zygote.gradient(omega -> get_added_mass(omega),1.5)


In [27]:
function get_added_mass2(omega)
    parameters = (wave_frequencies=[omega], 
    wave_directions=[betas[1]],
    radiating_dofs=[:Heave])
    mhresults = compute_and_label_hydrodynamic_coefficients(parameters, floatingbody1)
    added_mass = only(collect(mhresults.added_mass[wave_frequencies = At(omega)]))
    return added_mass
end
get_added_mass2(1.03)
# A_w_grad, = Zygote.gradient(omega -> get_added_mass2(omega),1.5)

7096.894912694628